# 06. 손 · 얼굴 점 찍기 — MediaPipe

MediaPipe 는 구글의 실시간 영상 AI 묶음이다. OpenVINO 가 아니라 자체 엔진(TFLite)으로 CPU 에서 돈다.
- **손**: 손마다 점 21개 + 손동작 이름(엄지척, 브이, 주먹 …)
- **얼굴**: 점 478개 (눈·입·홍채까지)

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

import mediapipe as mp
from mediapipe.tasks.python import BaseOptions, vision

MP = MODELS / "mediapipe"
hands = vision.GestureRecognizer.create_from_options(vision.GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path=str(MP / "gesture_recognizer.task")), num_hands=2))
face = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=str(MP / "face_landmarker.task")), num_faces=1))

def mp_image(bgr):
    return mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

## 2. 사진 — 손을 보여 주며 찍는다

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 손 점 21개 + 손동작
점 번호: 0 손목 · 1~4 엄지 · 5~8 검지 · 9~12 중지 · 13~16 약지 · 17~20 새끼 (각 손가락은 뿌리 → 끝).
좌표는 0~1 (사진 크기에 대한 비율)이라 픽셀로 바꿔서 그린다.

In [ ]:
FINGERS = [(0, 1, 2, 3, 4), (0, 5, 6, 7, 8), (5, 9, 10, 11, 12), (9, 13, 14, 15, 16), (13, 17, 18, 19, 20), (0, 17)]

res = hands.recognize(mp_image(frame))
h, w = frame.shape[:2]
view = frame.copy()
for i, lms in enumerate(res.hand_landmarks):
    pts = [(int(p.x * w), int(p.y * h)) for p in lms]
    for chain in FINGERS:
        for a, b in zip(chain, chain[1:]):
            cv2.line(view, pts[a], pts[b], (255, 200, 0), 2)
    for x, y in pts:
        cv2.circle(view, (x, y), 4, (0, 0, 255), -1)
    g = res.gestures[i][0] if res.gestures and res.gestures[i] else None
    side = res.handedness[i][0].category_name if res.handedness else "?"
    print(f"손 {i}: {side} · 손동작 {g.category_name if g else '-'} ({g.score:.2f})" if g else f"손 {i}: {side}")
print("찾은 손:", len(res.hand_landmarks))
show(view)

## 4. 얼굴 점 478개

In [ ]:
r = face.detect(mp_image(frame))
view = frame.copy()
for lms in r.face_landmarks:
    for p in lms:
        cv2.circle(view, (int(p.x * w), int(p.y * h)), 1, (0, 255, 120), -1)
    print("점 수:", len(lms))
show(view)

## 해 볼 것
1. 손가락이 몇 개 펴졌는지 세 보자 — 손가락 끝(4, 8, 12, 16, 20)이 두 번째 마디(3, 6, 10, 14, 18)보다 위(y 가 작다)면 편 것.
   (엄지는 x 로 비교한다.)
2. 엄지 끝(4)과 검지 끝(8) 사이 거리로 "집게" 를 판정해 보자.
3. 얼굴 점 13(윗입술)과 14(아랫입술) 사이로 입이 열렸는지 판정해 보자.
4. 이 결과로 07 번처럼 나만의 손동작 분류기를 만들 수 있다 — 점 21개 × (x, y, z) = 숫자 63개를 특징으로 쓴다.